# SEC EDGAR Filing Fetcher
Enter your CIK codes, date range, and form types below, then run each cell in order.

The **Escrow & Contingent Share Detection** section at the bottom is optional — 
run it after the fetch to scan filings for locked/restricted share arrangements.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
from dataclasses import asdict

from sec_filings_sync import fetch_filings_for_ciks, SECBlockedError


In [2]:
# ── User Inputs ───────────────────────────────────────────────────────────────
# Add or remove CIK codes here (strings or ints both work)
CIKS = [
    "320193",    # Apple
    "789019",    # Microsoft
    "1018724",   # Amazon
    "1652044",   # Alphabet
    "1823584",   # Alliance Entertainment (AENT) — known escrow structure
    "1830081",
]

# Filter to specific form types, or set to None to return everything
FORM_TYPES = ["10-K", "10-Q", "8-K"]

# Date range — both are inclusive. Set either to None for an open bound.
START_DATE = "2025-04-30"   # "YYYY-MM-DD"
END_DATE   = "2026-04-30"   # "YYYY-MM-DD"

# ── Request delay ─────────────────────────────────────────────────────────────
# Seconds to pause between each HTTP request sent to the SEC.
# The SEC fair-access policy caps automated requests at 10 per second.
# 0.11 s => ~9 req/s, leaving a small buffer below that cap.
# Raise this value (e.g. DELAY = 0.5) if you see 429 rate-limit errors.
DELAY = 0.11

print(f"CIKs       : {CIKS}")
print(f"Form types : {FORM_TYPES}")
print(f"Date range : {START_DATE}  ->  {END_DATE}")
print(f"Delay      : {DELAY} s/request  (~{1/DELAY:.0f} req/s, SEC cap is 10 req/s)")


CIKs       : ['320193', '789019', '1018724', '1652044', '1823584', '1830081']
Form types : ['10-K', '10-Q', '8-K']
Date range : 2025-04-30  ->  2026-04-30
Delay      : 0.11 s/request  (~9 req/s, SEC cap is 10 req/s)


---
## Synchronous fetch
Processes each CIK one at a time with a `DELAY`-second pause between requests.
Progress is printed as each CIK completes.

In [3]:
try:
    sync_filings = fetch_filings_for_ciks(
        ciks       = CIKS,
        form_types = FORM_TYPES,
        start_date = START_DATE,
        end_date   = END_DATE,
        delay      = DELAY,
    )
    print(f'\nTotal filings returned: {len(sync_filings)}')

except SECBlockedError as e:
    sync_filings = []
    print()
    print('=' * 60)
    print('SEC TIMEOUT — rate limit exceeded.')
    print(str(e))
    print('Wait 10 minutes, then re-run from this cell.')
    print('=' * 60)


[sync] 0000320193  Apple Inc.                             16 filings matched
[sync] 0000789019  MICROSOFT CORP                         13 filings matched
[sync] 0001018724  AMAZON COM INC                         17 filings matched
[sync] 0001652044  Alphabet Inc.                          19 filings matched
[sync] 0001823584  ALLIANCE ENTERTAINMENT HOLDING CORP    10 filings matched
[sync] 0001830081  Rumble Inc.                            13 filings matched

Total filings returned: 88


In [4]:
# Display as a DataFrame
sync_df = pd.DataFrame([asdict(f) for f in sync_filings])
sync_df


,cik,entity_name,ticker,form_type,filing_date,report_date,accession_number,primary_document,filing_url,index_url,file_number,act,size,is_xbrl
0,320193,Apple Inc.,AAPL,8-K,2026-04-30,2026-04-30,0000320193-26-000011,aapl-20260430.htm,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...,001-36743,34,412047,True
1,1018724,AMAZON COM INC,AMZN,10-Q,2026-04-30,2026-03-31,0001018724-26-000014,amzn-20260331.htm,https://www.sec.gov/Archives/edgar/data/101872...,https://www.sec.gov/Archives/edgar/data/101872...,001-43202,34,8488684,True
2,1652044,Alphabet Inc.,GOOGL,10-Q,2026-04-30,2026-03-31,0001652044-26-000048,goog-20260331.htm,https://www.sec.gov/Archives/edgar/data/165204...,https://www.sec.gov/Archives/edgar/data/165204...,001-37580,34,12460815,True
3,789019,MICROSOFT CORP,MSFT,10-Q,2026-04-29,2026-03-31,0001193125-26-191507,msft-20260331.htm,https://www.sec.gov/Archives/edgar/data/789019...,https://www.sec.gov/Archives/edgar/data/789019...,001-37845,34,31304294,True
4,789019,MICROSOFT CORP,MSFT,8-K,2026-04-29,2026-04-29,0001193125-26-191457,msft-20260429.htm,https://www.sec.gov/Archives/edgar/data/789019...,https://www.sec.gov/Archives/edgar/data/789019...,001-37845,34,1080617,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,320193,Apple Inc.,AAPL,8-K,2025-05-01,2025-05-01,0000320193-25-000055,aapl-20250501.htm,https://www.sec.gov/Archives/edgar/data/320193...,https://www.sec.gov/Archives/edgar/data/320193...,001-36743,34,436093,True
84,1018724,AMAZON COM INC,AMZN,8-K,2025-05-01,2025-05-01,0001018724-25-000034,amzn-20250501.htm,https://www.sec.gov/Archives/edgar/data/101872...,https://www.sec.gov/Archives/edgar/data/101872...,000-22513,34,923692,True
85,1652044,Alphabet Inc.,GOOGL,8-K,2025-05-01,2025-05-01,0001193125-25-110020,d884388d8k.htm,https://www.sec.gov/Archives/edgar/data/165204...,https://www.sec.gov/Archives/edgar/data/165204...,001-37580,34,499317,True
86,789019,MICROSOFT CORP,MSFT,10-Q,2025-04-30,2025-03-31,0000950170-25-061046,msft-20250331.htm,https://www.sec.gov/Archives/edgar/data/789019...,https://www.sec.gov/Archives/edgar/data/789019...,001-37845,34,29845298,True


---
## Quick checks

In [5]:
# Filings per company
sync_df.groupby(["cik", "entity_name"])["form_type"].count().rename("filings").reset_index()


,cik,entity_name,filings
0,1018724,AMAZON COM INC,17
1,1652044,Alphabet Inc.,19
2,1823584,ALLIANCE ENTERTAINMENT HOLDING CORP,10
3,1830081,Rumble Inc.,13
4,320193,Apple Inc.,16
5,789019,MICROSOFT CORP,13


In [6]:
# Filings per form type
sync_df["form_type"].value_counts().reset_index().rename(columns={"count": "filings"})


,form_type,filings
0,8-K,62
1,10-Q,20
2,10-K,6


In [7]:
# Filing URLs only — easy to copy out
sync_df[["entity_name", "ticker", "form_type", "filing_date", "filing_url"]]


,entity_name,ticker,form_type,filing_date,filing_url
0,Apple Inc.,AAPL,8-K,2026-04-30,https://www.sec.gov/Archives/edgar/data/320193...
1,AMAZON COM INC,AMZN,10-Q,2026-04-30,https://www.sec.gov/Archives/edgar/data/101872...
2,Alphabet Inc.,GOOGL,10-Q,2026-04-30,https://www.sec.gov/Archives/edgar/data/165204...
3,MICROSOFT CORP,MSFT,10-Q,2026-04-29,https://www.sec.gov/Archives/edgar/data/789019...
4,MICROSOFT CORP,MSFT,8-K,2026-04-29,https://www.sec.gov/Archives/edgar/data/789019...
...,...,...,...,...,...
83,Apple Inc.,AAPL,8-K,2025-05-01,https://www.sec.gov/Archives/edgar/data/320193...
84,AMAZON COM INC,AMZN,8-K,2025-05-01,https://www.sec.gov/Archives/edgar/data/101872...
85,Alphabet Inc.,GOOGL,8-K,2025-05-01,https://www.sec.gov/Archives/edgar/data/165204...
86,MICROSOFT CORP,MSFT,10-Q,2025-04-30,https://www.sec.gov/Archives/edgar/data/789019...


---
## Optional — Escrow & contingent share detection

Uses `escrow_shares.py` to scan each fetched filing for shares held in escrow,
earnout shares, founder shares, and similar locked / restricted arrangements.

**How it works**
1. Fetches the primary filing document (up to 800 KB) for each FilingRecord.
2. Strips HTML to plain text.
3. Scans for trigger keywords: *escrow*, *contingent shares*, *earn-out shares*,
   *founder shares*, *performance shares*, *milestone shares*.
4. For each hit, opens a context window and finds the nearest share count using
   proximity-anchored matching anchored to the placement verb
   (*placed in escrow*, *held in escrow*, *deposited into escrow*, etc.).
5. Validates: a bare mention of *escrow* with no placement verb is discarded
   to cut false positives.
6. Classifies the type (earnout → founder → performance → lockup → contingent
   → escrow → general) and extracts a trigger-hint fragment.

Returns **one row per distinct finding** per CIK.

| `escrow_type` | Meaning |
|---------------|---------|
| `earnout`     | Earn-out / earnout shares |
| `founder`     | Founder shares subject to forfeiture |
| `performance` | Performance or milestone shares |
| `lockup`      | Lock-up arrangement |
| `contingent`  | SPAC / merger contingent shares |
| `escrow`      | Explicit placement verb, no named type |
| `general`     | Keyword match, no more specific classifier |

In [8]:
from escrow_shares import find_escrow_shares

print('escrow_shares imported OK')


escrow_shares imported OK


### Run detection

Passes the filings list fetched above into `find_escrow_shares`.
Reuses a single HTTP session and de-duplicates by `(cik, shares, class)`
so the same escrow event in multiple filings appears only once per company.

In [9]:
if not sync_filings:
    print('No filings to scan — fetch was blocked or returned nothing.')
else:
    try:
        df_escrow = find_escrow_shares(sync_filings, delay=DELAY)
        print(f'\nTotal findings: {len(df_escrow)}')

    except SECBlockedError as e:
        df_escrow = __import__('pandas').DataFrame(columns=[
            'cik', 'entity_name', 'form_type', 'filing_date',
            'shares_in_escrow', 'share_class', 'escrow_type',
            'trigger_hint', 'source_text',
        ])
        print()
        print('=' * 60)
        print('SEC TIMEOUT — rate limit exceeded during escrow scan.')
        print(str(e))
        print('Wait 10 minutes, then re-run from this cell.')
        print('=' * 60)


  [escrow] Apple Inc.                     8-K    2026-04-30  findings=0
  [escrow] AMAZON COM INC                 10-Q   2026-04-30  findings=0
  [escrow] Alphabet Inc.                  10-Q   2026-04-30  findings=0
  [escrow] MICROSOFT CORP                 10-Q   2026-04-29  findings=0
  [escrow] MICROSOFT CORP                 8-K    2026-04-29  findings=0
  [escrow] AMAZON COM INC                 8-K    2026-04-29  findings=0
  [escrow] Alphabet Inc.                  8-K    2026-04-29  findings=0
  [escrow] Apple Inc.                     8-K    2026-04-20  findings=0
  [escrow] AMAZON COM INC                 8-K    2026-04-14  findings=0
  [escrow] Alphabet Inc.                  8-K    2026-04-10  findings=0
  [escrow] AMAZON COM INC                 8-K    2026-04-09  findings=0
  [escrow] Alphabet Inc.                  8-K    2026-04-02  findings=0
  [escrow] Rumble Inc.                    8-K    2026-03-27  findings=0
  [escrow] AMAZON COM INC                 8-K    2026-03-16  fin

### Results — all findings

In [10]:
if df_escrow.empty:
    print('No escrow findings in this filing set.')
else:
    display_df = df_escrow.copy()
    display_df['shares_in_escrow'] = display_df['shares_in_escrow'].apply(
        lambda x: f'{int(x):,}' if not __import__('pandas').isna(x) else 'n/a'
    )
    display_df['filing_date'] = display_df['filing_date'].astype(str).str[:10]
    display(display_df[[
        'entity_name', 'form_type', 'filing_date',
        'share_class', 'shares_in_escrow', 'escrow_type', 'trigger_hint',
    ]])


,entity_name,form_type,filing_date,share_class,shares_in_escrow,escrow_type,trigger_hint
0,Rumble Inc.,10-Q,2025-11-10,Common Stock,"28,587,396",escrow,
1,Rumble Inc.,10-Q,2025-11-10,Resulting In A Per,"2,027",earnout,"released at each target, or if the latter targ..."
2,ALLIANCE ENTERTAINMENT HOLDING CORP,10-K,2025-09-10,Class E Common Stock Of Alliance,"60,000,000",contingent,to be released to such Legacy Alliance stockho...
3,ALLIANCE ENTERTAINMENT HOLDING CORP,10-Q,2025-05-15,Class E Common Stock Of Adara,"60,000,000",contingent,to be released to such Legacy Alliance stockho...


### Source text review

Prints the matched passage (up to 500 chars) for each finding so results
can be manually verified against the original filing language.

In [11]:
for _, row in df_escrow.iterrows():
    print('=' * 80)
    print(f"  {row['entity_name']}  |  {row['form_type']}  |  {str(row['filing_date'])[:10]}")
    print(f"  Class    : {row['share_class']}")
    shares_str = f"{int(row['shares_in_escrow']):,}" if not __import__('pandas').isna(row['shares_in_escrow']) else 'n/a'
    print(f"  Shares   : {shares_str}")
    print(f"  Type     : {row['escrow_type']}")
    print(f"  Trigger  : {row['trigger_hint']}")
    print()
    print(row['source_text'])
    print()

if df_escrow.empty:
    print('No findings to review.')


  Rumble Inc.  |  10-Q  |  2025-11-10
  Class    : Common Stock
  Shares   : 28,587,396
  Type     : escrow
  Trigger  : 

24, respectively: As of September 30, 2025 As of December 31, 2024 Number Amount Number Amount Class A Common Stock 215,380,893 $ 751,456 118,808,857 $ 741,799 Class C Common Stock (and its corresponding ExchangeCo Share) 123,690,470 12,369 165,153,621 16,515 Class D Common Stock 95,791,120 9,579 105,782,403 10,578 Balance 434,862,483 $ 773,404 389,744,881 $ 768,892 Former
holders of Legacy Rumble’s (as defined below) common shares are eligible to receive up to an aggregate of 105,000,000 additi

  Rumble Inc.  |  10-Q  |  2025-11-10
  Class    : Resulting In A Per
  Shares   : 2,027
  Type     : earnout
  Trigger  : released at each target, or if the latter target is reached first, 100 %) for a period of 20 trading

en the contingency is met.
The holders are eligible to the shares if the closing price of the Company’s Class A Common Stock is greater than or equal 